# Looped transformers: общие веса

Повторное применение ядра, глубина вычислений и backward через циклы.

> Все примеры этого урока работают на CPU, не скачивают модели и используют искусственные данные. Они объясняют механизм; качество настоящего обучения проверяется отдельно.

## Что важно понять

Looped transformer применяет одно и то же ядро несколько раз. Число уникальных параметров не растёт с количеством проходов, но вычисления и память активаций обычно растут. Списки из независимых копий блока не дают sharing.

Ниже общее causal-attention ядро. Full backprop проходит через все итерации. Detach между проходами обрезает градиент и меняет обучение; checkpointing пересчитывает активации и при корректной реализации сохраняет градиент. Изменение loop depth на inference — эксперимент, а не гарантированное улучшение. В Builder эти параметры задаются у looped блока.

In [ ]:
%matplotlib inline
import torch
from torch import nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(42)
plt.rcParams.update({'figure.figsize': (7, 3), 'axes.spines.top': False, 'axes.spines.right': False})
print('PyTorch:', torch.__version__, '| device: CPU')

## Запустите эксперимент

Запускайте ячейки по порядку. Код можно менять; Shift + Enter выполняет текущую ячейку.

In [ ]:
class Looped(nn.Module):
    def __init__(self):
        super().__init__()
        self.core = nn.TransformerEncoderLayer(16, 4, 32, dropout=0, batch_first=True)
    def forward(self, h, loops):
        mask = torch.triu(torch.ones(h.shape[1], h.shape[1], dtype=torch.bool), diagonal=1)
        for _ in range(loops):
            h = self.core(h, src_mask=mask)
        return h
model = Looped(); x = torch.randn(2, 6, 16)
parameters = sum(p.numel() for p in model.parameters())
for depth in [1, 2, 4]:
    model.zero_grad(set_to_none=True)
    output = model(x, depth)
    # LayerNorm makes output.square().mean() nearly constant; use a nonuniform target.
    loss = F.mse_loss(output, x * 0.3)
    loss.backward()
    print('loops:', depth, 'unique parameters:', parameters, 'core gradient:', model.core.linear1.weight.grad.norm().item())

## Попробуйте сами

Сравните с ModuleList из четырёх независимых ядер. Проверьте число параметров и id(weight). Почему равные начальные значения весов ещё не означают общие параметры?

Сохраните копию ноутбука и запишите, что изменили, чего ожидали и что измерили.

## Продолжить в студии

Кнопка «Открыть раздел студии» над уроком ведёт к связанным инструментам. Учебный код сам не запускает большие тренировки. Перед реальным запуском проверьте данные, ресурсы и поддержку метода у выбранной модели.

## Источники

- [Universal Transformers](https://arxiv.org/abs/1807.03819)
- [ALBERT](https://arxiv.org/abs/1909.11942)